# P4-6 · Multi-agent

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q langgraph

## Lý thuyết

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    task: str
    findings: str
    analysis: str
    report: str
    next: str

# worker giả lập để chạy thử đồ thị; trong hệ thật mỗi hàm là một agent có LLM và tool riêng
def researcher(s): print("researcher làm việc"); return {"findings": "3 đối thủ, giá 2–4 triệu"}
def analyst(s):    print("analyst làm việc");    return {"analysis": "giá shop ta ở giữa"}
def writer(s):     print("writer làm việc");     return {"report": "Báo cáo: ..."}

def supervisor(s):
    if not s.get("findings"): return {"next": "researcher"}
    if not s.get("analysis"): return {"next": "analyst"}
    if not s.get("report"): return {"next": "writer"}
    return {"next": "done"}

g = StateGraph(State)
g.add_node("supervisor", supervisor)
g.add_node("researcher", researcher)      # mỗi worker là một agent riêng (có tool riêng)
g.add_node("analyst", analyst)
g.add_node("writer", writer)
g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", lambda s: s["next"],
                        {"researcher": "researcher", "analyst": "analyst", "writer": "writer", "done": END})
for w in ["researcher", "analyst", "writer"]:
    g.add_edge(w, "supervisor")
app = g.compile()
app.invoke({"task": "So sánh giá bàn trang điểm với đối thủ"})["report"]

researcher làm việc
analyst làm việc
writer làm việc
'Báo cáo: ...'


In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class State(TypedDict):
    competitors: list[str]
    results: Annotated[list, operator.add]        # mỗi worker thêm phần của mình
    report: str

class WorkerInput(TypedDict):
    name: str

def fan_out(s: State):
    return [Send("research_one", {"name": c}) for c in s["competitors"]]

def research_agent(name):                          # giả lập; hệ thật: một agent nhỏ có tool tìm kiếm
    return f"{name}: giá từ {len(name)} triệu"

def writer_llm(results):                           # giả lập; hệ thật: một lần gọi LLM tổng hợp
    return " | ".join(r["summary"] for r in results)

def research_one(w: WorkerInput):
    summary = research_agent(w["name"])            # một agent nhỏ, context riêng
    return {"results": [{"name": w["name"], "summary": summary}]}

def write(s: State):
    return {"report": writer_llm(s["results"])}

g = StateGraph(State)
g.add_node("research_one", research_one)
g.add_node("write", write)
g.add_conditional_edges(START, fan_out, ["research_one"])
g.add_edge("research_one", "write")
g.add_edge("write", END)
app = g.compile()
app.invoke({"competitors": ["Hòa Bình", "An Phát", "Gỗ Việt"], "results": []})

{'competitors': ['Hòa Bình', 'An Phát', 'Gỗ Việt'], 'results': [{'name': 'Hòa Bình', 'summary': 'Hòa Bình: giá từ 8 triệu'}, {'name': 'An Phát', 'summary': 'An Phát: giá từ 7 triệu'}, {'name': 'Gỗ Việt', 'summary': 'Gỗ Việt: giá từ 7 triệu'}], 'report': 'Hòa Bình: giá từ 8 triệu | An Phát: giá từ 7 triệu | Gỗ Việt: giá từ 7 triệu'}


## Bài tập 5: Fan-out song song {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class State(TypedDict):
    competitors: list[str]
    findings: Annotated[list, operator.add]     # các nhánh song song cùng ghi vào, nối danh sách lại
    report: str

def fan_out(state):
    return [Send("research_one", {"name": c}) for c in state["competitors"]]

def research_one(arg):
    summary = research_agent(arg["name"])       # agent con với tool tìm kiếm, trả bản tóm tắt
    return {"findings": [{"name": arg["name"], "summary": summary}]}

def write(state):
    return {"report": writer_agent(state["findings"])}

g = StateGraph(State)
g.add_node("research_one", research_one)
g.add_node("write", write)
g.add_conditional_edges(START, fan_out, ["research_one"])
g.add_edge("research_one", "write")
g.add_edge("write", END)
app = g.compile()

## Bài tập 6: Supervisor định tuyến yêu cầu {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
TU_KHOA = {"don_hang": ["đơn", "giao", "ship", "hủy"], "ky_thuat": ["lỗi", "đăng nhập", "app", "mật khẩu"],
           "ke_toan": ["hóa đơn", "hoàn tiền", "thanh toán", "vat"]}
mau = [("Đơn DH12 giao chưa", "don_hang"), ("App báo lỗi khi mở", "ky_thuat"), ("Xuất hóa đơn VAT giúp mình", "ke_toan"),
       ("Quên mật khẩu", "ky_thuat"), ("Muốn hủy đơn hôm qua", "don_hang"), ("Bao giờ được hoàn tiền", "ke_toan"),
       ("Thanh toán lỗi, bị trừ tiền mà đơn chưa tạo", "ke_toan"), ("Shipper gọi không được", "don_hang"),
       ("Không đăng nhập được app", "ky_thuat"), ("Hóa đơn của đơn hàng DH9 sai tên công ty", "ke_toan")]

def dinh_tuyen(yeu_cau):
    # TODO: đếm số từ khóa mỗi agent xuất hiện trong yêu cầu (chữ thường), chọn agent nhiều nhất
    return "hoi_lai"

In [ ]:
def dinh_tuyen(yeu_cau):
    diem = {ag: sum(k in yeu_cau.lower() for k in ks) for ag, ks in TU_KHOA.items()}
    return "hoi_lai" if max(diem.values()) == 0 else max(diem, key=diem.get)

sai = [(y, dinh_tuyen(y), nhan) for y, nhan in mau if dinh_tuyen(y) != nhan]
acc = 1 - len(sai) / len(mau)
print("độ chính xác:", acc)
for y, du_doan, nhan in sai:
    print(f"  nhầm: '{y}' → {du_doan} (đúng: {nhan})")

độ chính xác: 0.8
  nhầm: 'Thanh toán lỗi, bị trừ tiền mà đơn chưa tạo' → don_hang (đúng: ke_toan)
  nhầm: 'Hóa đơn của đơn hàng DH9 sai tên công ty' → don_hang (đúng: ke_toan)


## Bài tập 7: Chuyển giao bằng tóm tắt thay vì cả hội thoại {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json
hoi_thoai = []
for i in range(6):
    hoi_thoai.append({"role": "user", "content": "Dạ em chào shop, em hỏi chút ạ, cái app của shop dạo này hay bị sao ấy, em dùng điện thoại Samsung A52, Android 13."})
    hoi_thoai.append({"role": "assistant", "content": "Dạ shop chào anh/chị, anh/chị mô tả giúp shop lỗi cụ thể và mã lỗi hiển thị (nếu có) được không ạ?"})
hoi_thoai[-2]["content"] = "Mã lỗi E-401 khi đăng nhập, tài khoản lan.nguyen, từ hôm qua, đã thử cài lại app rồi."

def uoc_token(x):
    return len(x) // 3 + 1
# TODO: token_a; trang_thai (dict có loi, ma_loi, thiet_bi, tai_khoan, da_thu); token_b

In [ ]:
token_a = uoc_token(json.dumps(hoi_thoai, ensure_ascii=False))
trang_thai = {"loi": "không đăng nhập được app", "ma_loi": "E-401", "thiet_bi": "Samsung A52, Android 13",
              "tai_khoan": "lan.nguyen", "tu_khi": "hôm qua", "da_thu": ["cài lại app"]}
token_b = uoc_token(json.dumps(trang_thai, ensure_ascii=False))
print(f"(a) {token_a} token | (b) {token_b} token, tiết kiệm {1 - token_b / token_a:.0%}")

(a) 559 token | (b) 57 token, tiết kiệm 90%


## Bài tập 8: Supervisor bằng LangGraph {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    so_lieu: list
    bao_cao: str
    luot: int
    nhat_ky: list

NGUON = ["Doanh thu Q3 tăng 12%", "Tỉ lệ hoàn đơn giảm còn 4%", "Đơn từ app chiếm 55%", "Chi phí quảng cáo tăng 8%"]

def nghien_cuu(s):
    return {"so_lieu": s["so_lieu"] + [NGUON[len(s["so_lieu"])]], "nhat_ky": s["nhat_ky"] + ["nghien_cuu"]}

def viet(s):
    return {"bao_cao": "Báo cáo: " + "; ".join(s["so_lieu"]) + ".", "nhat_ky": s["nhat_ky"] + ["viet"]}

def supervisor(s):
    return {"luot": s["luot"] + 1, "nhat_ky": s["nhat_ky"] + ["supervisor"]}
# TODO: hàm chon(s) trả về "nghien_cuu" | "viet" | END; build graph; kq = app.invoke(...)

In [ ]:
def chon(s):
    if s["luot"] > 8:
        return END
    if len(s["so_lieu"]) < 3:
        return "nghien_cuu"
    return "viet" if not s["bao_cao"] else END

g = StateGraph(State)
g.add_node("supervisor", supervisor); g.add_node("nghien_cuu", nghien_cuu); g.add_node("viet", viet)
g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", chon)
g.add_edge("nghien_cuu", "supervisor"); g.add_edge("viet", "supervisor")
app = g.compile()
kq = app.invoke({"so_lieu": [], "bao_cao": "", "luot": 0, "nhat_ky": []})
print(" → ".join(kq["nhat_ky"]))
print(kq["bao_cao"])

supervisor → nghien_cuu → supervisor → nghien_cuu → supervisor → nghien_cuu → supervisor → viet → supervisor
Báo cáo: Doanh thu Q3 tăng 12%; Tỉ lệ hoàn đơn giảm còn 4%; Đơn từ app chiếm 55%.
